# MEMS pressure sensor: re-run the main results
Upload to Google Colab and use Runtime > Run all. The first cell writes the model scripts and the input tables (same files as in the repository), the next cells re-run the checks. Expected numbers are written above each cell.

In [ ]:
import os
!pip -q install scikit-fem==12.0.2
FILES = {'membrane_fem.py': 'import argparse, json, time\nimport numpy as np\ntry:\n    from pypardiso import spsolve\n    fmt = "csr"\nexcept ImportError:\n    from scipy.sparse.linalg import spsolve\n    fmt = "csc"\nfrom skfem import (MeshHex, Basis, ElementHex2, ElementVector, BilinearForm, LinearForm,\n                   FacetBasis, solve, condense)\n\np = argparse.ArgumentParser()\np.add_argument("--side", type=float, default=600.0)\np.add_argument("--t", type=float, default=13.0)\np.add_argument("--p", type=float, default=100.0)\np.add_argument("--nxy", type=int, default=16)\np.add_argument("--nz", type=int, default=2)\np.add_argument("--iso", type=float, nargs=2, default=None)\np.add_argument("--pi", type=float, nargs=3, default=[6.6, -1.1, 138.1])\np.add_argument("--resL", type=float, nargs=3, default=[30, 5, 5])\np.add_argument("--resT", type=float, nargs=3, default=[30, 5, 5])\np.add_argument("--depth", type=float, default=0.0)\np.add_argument("--frame", type=float, nargs=2, default=None)\np.add_argument("--map", type=str, default=None)\np.add_argument("--profile", type=str, default=None)\np.add_argument("--out", type=str, default=None)\nargs = p.parse_args()\n\nt0 = time.time()\num = 1e-6\na = args.side / 2 * um\nT = args.t * um\nP = args.p * 1e3\n\ndef cubic_tensor(c11, c12, c44):\n    C = np.zeros((3, 3, 3, 3))\n    for i in range(3):\n        for j in range(3):\n            for k in range(3):\n                for l in range(3):\n                    if i == j and k == l:\n                        C[i, j, k, l] = c11 if i == k else c12\n                    elif (i == k and j == l) or (i == l and j == k):\n                        if i != j:\n                            C[i, j, k, l] = c44\n    return C\n\ndef rotate(C, ang):\n    c, s = np.cos(ang), np.sin(ang)\n    R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])\n    return np.einsum(\'ia,jb,kc,ld,abcd->ijkl\', R, R, R, R, C)\n\nif args.iso:\n    E, nu = args.iso[0] * 1e9, args.iso[1]\n    lam, mu = E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))\n    Ct = cubic_tensor(lam + 2 * mu, lam, mu)\n    material = f"isotropic E={args.iso[0]} GPa nu={nu}"\nelse:\n    Ct = rotate(cubic_tensor(165.7e9, 63.9e9, 79.6e9), np.pi / 4)\n    material = "anisotropic Si (100), edges along <110>"\n\ns = np.linspace(0, 1, args.nxy + 1)\nxs = a * np.sin(np.pi / 2 * s)\nzs = np.linspace(0, T, args.nz + 1)\nif args.frame:\n    F, H = args.frame[0] * um, args.frame[1] * um\n    nf = max(4, args.nxy // 3)\n    xf = a + F * (np.linspace(0, 1, nf + 1)[1:] ** 1.6)\n    xs = np.concatenate([xs, xf])\n    zb = -H * (np.linspace(0, 1, nf + 1)[1:] ** 1.6)\n    zs = np.concatenate([zb[::-1], zs])\n    m = MeshHex.init_tensor(xs, xs, zs)\n    c = m.p[:, m.t].mean(axis=1)\n    cavity = np.nonzero((c[0] < a) & (c[1] < a) & (c[2] < 0))[0]\n    m = m.remove_elements(cavity)\nelse:\n    m = MeshHex.init_tensor(xs, xs, zs)\ne = ElementVector(ElementHex2())\nbasis = Basis(m, e)\n\nNZ_C = [(i, j, k, l, Ct[i, j, k, l]) for i in range(3) for j in range(3) for k in range(3)\n        for l in range(3) if abs(Ct[i, j, k, l]) > 1e-6 * abs(Ct).max()]\n\n@BilinearForm\ndef stiffness(u, v, w):\n    gu, gv = u.grad, v.grad\n    out = 0.0\n    for i, j, k, l, c in NZ_C:\n        out = out + c * gu[k, l] * gv[i, j]\n    return out\n\n@LinearForm\ndef load(v, w):\n    return -P * v.value[2]\n\nK = stiffness.assemble(basis)\ntop = m.facets_satisfying(lambda x: np.isclose(x[2], T))\nf = load.assemble(FacetBasis(m, e, facets=top))\n\nD = basis.get_dofs\nif args.frame:\n    xmax = xs[-1]\n    outer = D(m.facets_satisfying(lambda x: np.isclose(x[2], zs[0]))).all()\n    sides = np.concatenate([D(m.facets_satisfying(lambda x: np.isclose(x[0], xmax))).all(\'u^1\'),\n                            D(m.facets_satisfying(lambda x: np.isclose(x[1], xmax))).all(\'u^2\')])\n    clamp = np.concatenate([outer, sides])\nelse:\n    clamp = D(m.facets_satisfying(lambda x: np.isclose(x[0], a) | np.isclose(x[1], a))).all()\nfixed = np.unique(np.concatenate([\n    D(m.facets_satisfying(lambda x: np.isclose(x[0], 0))).all(\'u^1\'),\n    D(m.facets_satisfying(lambda x: np.isclose(x[1], 0))).all(\'u^2\'),\n    clamp,\n]))\n\ndef linsolve(A, b):\n    return spsolve(A.asformat(fmt), b)\n\nu = solve(*condense(K, f, D=fixed), solver=linsolve)\n\ncdofs = basis.get_dofs(nodes=lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)).nodal[\'u^3\']\nw_centre = float(np.mean(np.abs(u[cdofs])))\nedofs = basis.get_dofs(nodes=lambda x: np.isclose(x[0], a) & np.isclose(x[1], 0) & np.isclose(x[2], T)).nodal[\'u^3\']\nw_centre -= float(np.abs(u[edofs]).mean())\n\nsb = basis.with_element(ElementHex2())\nuh = basis.interpolate(u)\nstrain = 0.5 * (uh.grad + np.swapaxes(uh.grad, 0, 1))\nsig = np.einsum(\'ijkl,kl...->ij...\', Ct, strain)\n\n@BilinearForm\ndef mass(u_, v_, w):\n    return u_ * v_\n\n@LinearForm\ndef rhs(v_, w):\n    return w[\'s\'] * v_\n\nM = mass.assemble(sb)\nnodal = {}\nfor (i, j), name in {(0, 0): \'sxx\', (1, 1): \'syy\', (0, 1): \'sxy\'}.items():\n    b = rhs.assemble(sb, s=sig[i, j])\n    nodal[name] = linsolve(M, b)\n\ndef probe(name, pts):\n    return sb.probes(pts) @ nodal[name]\n\nxline = np.linspace(0, a, 601)\npts = np.vstack([xline, np.zeros_like(xline), np.full_like(xline, T)])\nprof_sxx, prof_syy = probe(\'sxx\', pts), probe(\'syy\', pts)\nsig_edge = float(prof_sxx[-1])\nsig_centre = float(prof_sxx[0])\n\np11, p12, p44 = [v * 1e-11 for v in args.pi]\npl, pt = (p11 + p12 + p44) / 2, (p11 + p12 - p44) / 2\n\ndef avg_rect(x0, x1, y0, y1, n=25):\n    X, Y = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))\n    zs_ = [T] if args.depth <= 0 else list(T - np.linspace(0, args.depth * um, 5))\n    sx, sy = [], []\n    for zz in zs_:\n        q = np.vstack([X.ravel(), Y.ravel(), np.full(X.size, zz)])\n        sx.append(probe(\'sxx\', q).mean()); sy.append(probe(\'syy\', q).mean())\n    return float(np.mean(sx)), float(np.mean(sy))\n\nLL, WL, gL = [v * um for v in args.resL]\nLT, WT, gT = [v * um for v in args.resT]\nsxL, syL = avg_rect(a - gL - LL, a - gL, 0, WL / 2)\ndR_L = pl * sxL + pt * syL\nsxT, syT = avg_rect(a - gT - WT, a - gT, 0, LT / 2)\ndR_T = pl * syT + pt * sxT\nvout = (dR_L - dR_T) / (2 + dR_L + dR_T)\n\nres = dict(material=material, side_um=args.side, t_um=args.t, p_kPa=args.p, nxy=args.nxy, nz=args.nz,\n           dofs=int(K.shape[0]), w_centre_um=w_centre / um, w_over_t=w_centre / T,\n           sigma_edge_MPa=sig_edge / 1e6, sigma_centre_MPa=sig_centre / 1e6,\n           pi_1e11=args.pi, resL_um=args.resL, resT_um=args.resT, res_depth_um=args.depth,\n           frame_um=args.frame, sxL_MPa=sxL / 1e6, syL_MPa=syL / 1e6, sxT_MPa=sxT / 1e6, syT_MPa=syT / 1e6,\n           sigma_x_2um_in_MPa=float(np.interp(a - 2 * um, xline, prof_sxx)) / 1e6,\n           dR_L=dR_L, dR_T=dR_T, vout_mV_per_V=vout * 1e3,\n           sens_mV_per_V_per_kPa=vout * 1e3 / args.p, seconds=round(time.time() - t0, 1))\nprint(json.dumps(res))\nif args.out:\n    json.dump(res, open(args.out, "w"), indent=1)\nif args.map:\n    g = np.linspace(0, a, 61)\n    X, Y = np.meshgrid(g, g)\n    q = np.vstack([X.ravel(), Y.ravel(), np.full(X.size, T)])\n    np.savetxt(args.map, np.column_stack([X.ravel() / um, Y.ravel() / um, probe(\'sxx\', q) / 1e6, probe(\'syy\', q) / 1e6]),\n               delimiter=",", header="x_um,y_um,sigma_x_MPa,sigma_y_MPa", comments="", fmt="%.5g")\nif args.profile:\n    np.savetxt(args.profile, np.column_stack([xline / um, prof_sxx / 1e6, prof_syy / 1e6]), delimiter=",",\n               header="x_um_from_centre,sigma_x_MPa,sigma_y_MPa", comments="", fmt="%.5g")\n', 'nonlinear_check.py': 'import argparse, json, time\nimport numpy as np\ntry:\n    from pypardiso import spsolve\n    fmt = "csr"\nexcept ImportError:\n    from scipy.sparse.linalg import spsolve\n    fmt = "csc"\nfrom skfem import MeshHex, Basis, ElementHex2, ElementVector, BilinearForm, LinearForm, FacetBasis, solve, condense\n\nap = argparse.ArgumentParser()\nap.add_argument("--side", type=float, default=600.0); ap.add_argument("--t", type=float, default=13.0)\nap.add_argument("--p", type=float, default=100.0); ap.add_argument("--nxy", type=int, default=12)\nap.add_argument("--nz", type=int, default=2); ap.add_argument("--steps", type=int, default=4)\nap.add_argument("--pi", type=float, nargs=3, default=[6.6, -1.1, 138.1])\nap.add_argument("--resL", type=float, nargs=3, default=[30, 5, 5]); ap.add_argument("--resT", type=float, nargs=3, default=[30, 5, 5])\nargs = ap.parse_args()\nt0 = time.time(); um = 1e-6\na, T, P = args.side / 2 * um, args.t * um, args.p * 1e3\n\ndef cubic(c11, c12, c44):\n    C = np.zeros((3, 3, 3, 3))\n    for i in range(3):\n        for j in range(3):\n            for k in range(3):\n                for l in range(3):\n                    if i == j and k == l: C[i, j, k, l] = c11 if i == k else c12\n                    elif ((i == k and j == l) or (i == l and j == k)) and i != j: C[i, j, k, l] = c44\n    return C\nc, s_ = np.cos(np.pi / 4), np.sin(np.pi / 4)\nR = np.array([[c, -s_, 0], [s_, c, 0], [0, 0, 1]])\nCt = np.einsum(\'ia,jb,kc,ld,abcd->ijkl\', R, R, R, R, cubic(165.7e9, 63.9e9, 79.6e9))\nNZ = [(i, j, k, l, Ct[i, j, k, l]) for i in range(3) for j in range(3) for k in range(3) for l in range(3)\n      if abs(Ct[i, j, k, l]) > 1e-6 * abs(Ct).max()]\n\nxs = a * np.sin(np.pi / 2 * np.linspace(0, 1, args.nxy + 1))\nm = MeshHex.init_tensor(xs, xs, np.linspace(0, T, args.nz + 1))\ne = ElementVector(ElementHex2()); basis = Basis(m, e)\nI3 = np.eye(3)\n\ndef kin(G):\n    F = G + I3[:, :, None, None]\n    E = 0.5 * (np.einsum(\'ki...,kj...->ij...\', F, F) - I3[:, :, None, None])\n    S = np.zeros_like(E)\n    for i, j, k, l, cc in NZ: S[i, j] += cc * E[k, l]\n    return F, S\n\ndef CC(X):\n    out = np.zeros_like(X)\n    for i, j, k, l, cc in NZ: out[i, j] += cc * X[k, l]\n    return out\n\n@BilinearForm\ndef tangent(du, v, w):\n    F, S = w[\'F\'], w[\'S\']\n    gdu, gv = du.grad, v.grad\n    dE = 0.5 * (np.einsum(\'ki...,kj...->ij...\', gdu, F) + np.einsum(\'ki...,kj...->ij...\', F, gdu))\n    dS = CC(dE)\n    dP = np.einsum(\'ik...,kj...->ij...\', gdu, S) + np.einsum(\'ik...,kj...->ij...\', F, dS)\n    return np.einsum(\'ij...,ij...->...\', dP, gv)\n\n@LinearForm\ndef internal(v, w):\n    Pk = np.einsum(\'ik...,kj...->ij...\', w[\'F\'], w[\'S\'])\n    return np.einsum(\'ij...,ij...->...\', Pk, v.grad)\n\n@LinearForm\ndef load(v, w):\n    return -v.value[2]\n\nfb = FacetBasis(m, e, facets=m.facets_satisfying(lambda x: np.isclose(x[2], T)))\nf1 = load.assemble(fb)\nD = basis.get_dofs\nfixed = np.unique(np.concatenate([D(m.facets_satisfying(lambda x: np.isclose(x[0], 0))).all(\'u^1\'),\n                                  D(m.facets_satisfying(lambda x: np.isclose(x[1], 0))).all(\'u^2\'),\n                                  D(m.facets_satisfying(lambda x: np.isclose(x[0], a) | np.isclose(x[1], a))).all()]))\ndef lin(A, b):\n    return spsolve(A.asformat(fmt), b)\n\nu = np.zeros(basis.N); log = []\nfor step in range(1, args.steps + 1):\n    load_now = P * step / args.steps\n    for it in range(25):\n        F, S = kin(basis.interpolate(u).grad)\n        r = load_now * f1 - internal.assemble(basis, F=F, S=S)\n        free = np.setdiff1d(np.arange(basis.N), fixed)\n        res = np.linalg.norm(r[free]) / np.linalg.norm(load_now * f1[free])\n        if res < 1e-8: break\n        K = tangent.assemble(basis, F=F, S=S)\n        u += solve(*condense(K, r, D=fixed), solver=lin)\n    log.append((step, it, float(res)))\n\ncd = basis.get_dofs(nodes=lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)).nodal[\'u^3\']\nw0 = float(np.mean(np.abs(u[cd])))\nF, S = kin(basis.interpolate(u).grad)\nsb = basis.with_element(ElementHex2())\n@BilinearForm\ndef mass(p_, q_, w): return p_ * q_\n@LinearForm\ndef rhs(q_, w): return w[\'s\'] * q_\nM = mass.assemble(sb)\nnod = {n: lin(M, rhs.assemble(sb, s=S[i, j])) for (i, j), n in {(0, 0): \'sxx\', (1, 1): \'syy\'}.items()}\ndef avg(x0, x1, y0, y1, n=25):\n    X, Y = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))\n    q = np.vstack([X.ravel(), Y.ravel(), np.full(X.size, T)])\n    Pm = sb.probes(q); return (Pm @ nod[\'sxx\']).mean(), (Pm @ nod[\'syy\']).mean()\np11, p12, p44 = [v * 1e-11 for v in args.pi]; pl, pt = (p11 + p12 + p44) / 2, (p11 + p12 - p44) / 2\nLL, WL, gL = [v * um for v in args.resL]; LT, WT, gT = [v * um for v in args.resT]\nsxL, syL = avg(a - gL - LL, a - gL, 0, WL / 2); sxT, syT = avg(a - gT - WT, a - gT, 0, LT / 2)\naL, bT = pl * sxL + pt * syL, pl * syT + pt * sxT\nvout = (aL - bT) / (2 + aL + bT)\nprint(json.dumps(dict(model="nonlinear ideal clamp", side_um=args.side, t_um=args.t, p_kPa=args.p, nxy=args.nxy, nz=args.nz,\n                      w_centre_um=w0 / um, w_over_t=w0 / T, sxL_MPa=sxL / 1e6, vout_mV_per_V=vout * 1e3,\n                      newton_log=log, seconds=round(time.time() - t0, 1))))\n', 'process_cost.py': 'import numpy as np, csv, os, json\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nDATA = os.path.join(HERE, "data")\nos.makedirs(DATA, exist_ok=True)\nrng = np.random.default_rng(2026)\nN_MC = 200_000\nT0 = 13.0\n\nrows = list(csv.DictReader(open(os.path.join(DATA, "sweep_su.csv"))))\nt_tab = np.array([float(r["t_um"]) for r in rows])\nv_tab = np.array([float(r["vout_mV_per_V"]) for r in rows])\norder = np.argsort(t_tab); t_tab, v_tab = t_tab[order], v_tab[order]\nlogfit = np.polyfit(np.log(t_tab), np.log(v_tab), 1)\n\ndef sens(t):\n    t = np.asarray(t, float)\n    inside = (t >= t_tab[0]) & (t <= t_tab[-1])\n    out = np.exp(np.polyval(logfit, np.log(np.clip(t, 1e-3, None))))\n    out[inside] = np.exp(np.interp(np.log(t[inside]), np.log(t_tab), np.log(v_tab)))\n    return out\n\nS0 = float(sens(np.array([T0]))[0])\n\nPUB = os.path.join(DATA, "public")\nROUTES, TOL, DIE = {}, {}, {}\nfor r in csv.DictReader(open(os.path.join(PUB, "thickness_control.csv"), encoding="utf-8")):\n    ROUTES[r["route"]] = (float(r["sigma_base_um"]), float(r["sigma_conservative_um"]), r["source"])\n    TOL[r["route"]] = float(r["band_um"])\n    DIE[r["route"]] = float(r["die_mm"])\nUNPUBLISHED = {}\nfor r in csv.DictReader(open(os.path.join(PUB, "thickness_scenarios.csv"), encoding="utf-8")):\n    UNPUBLISHED.setdefault(r["route"], []).append(float(r["sigma_um"]))\nGAINS = [2, 5, 10, 15, 20, 25]\n\ndef share_in(sig, G):\n    t = T0 + rng.normal(0, sig, N_MC)\n    s = sens(t) / S0\n    return float(np.mean(np.abs(s - 1) <= G / 100)), float(np.std(s) / np.mean(s))\n\nproc_rows = []\nfor name, (sb, sc, src) in ROUTES.items():\n    for case, sig in (("base", sb), ("conservative", sc)):\n        r = {"route": name, "case": case, "sigma_um": round(sig, 4), "source": src}\n        for G in GAINS:\n            sh, cv = share_in(sig, G)\n            r[f"in_pm{G}pct"] = round(sh, 4)\n        r["sensitivity_CV_pct"] = round(100 * cv, 2)\n        proc_rows.append(r)\nfor name, sigs in UNPUBLISHED.items():\n    for sig in sigs:\n        r = {"route": name, "case": f"sigma={sig}", "sigma_um": sig, "source": "ASSUMPTION"}\n        for G in GAINS:\n            sh, cv = share_in(sig, G)\n            r[f"in_pm{G}pct"] = round(sh, 4)\n        r["sensitivity_CV_pct"] = round(100 * cv, 2)\n        proc_rows.append(r)\n\nwith open(os.path.join(DATA, "process_montecarlo.csv"), "w", newline="") as fh:\n    w = csv.DictWriter(fh, fieldnames=list(proc_rows[0].keys())); w.writeheader(); w.writerows(proc_rows)\n\nGs = np.round(np.geomspace(0.5, 30, 40), 3)\ncurve = [["G_pct"] + [n for n in ROUTES]]\nsamples = {n: sens(T0 + np.random.default_rng(7).normal(0, ROUTES[n][0], N_MC)) / S0 for n in ROUTES}\nfor G in Gs:\n    curve.append([G] + [round(float(np.mean(np.abs(samples[n] - 1) <= G / 100)), 4) for n in ROUTES])\nwith open(os.path.join(DATA, "process_curve_base.csv"), "w", newline="") as fh:\n    csv.writer(fh).writerows(curve)\n\nINPUTS = {}\nfor r in csv.DictReader(open(os.path.join(PUB, "cost_model_inputs.csv"), encoding="utf-8")):\n    INPUTS[r["name"]] = (float(r["base"]), float(r["low"]), float(r["high"]), r["source"] or "ASSUMPTION")\ndef n_gross(die_mm, street=0.1, wafer=150.0, edge=3.0):\n    D, A = wafer - 2 * edge, (die_mm + street) ** 2\n    return int(np.pi * (D / 2) ** 2 / A - np.pi * D / np.sqrt(2 * A))\nN_GROSS = n_gross(1.2)\n\ndef y_route(sig, G):\n    t = T0 + np.random.default_rng(11).normal(0, sig, N_MC)\n    return float(np.mean(np.abs(sens(t) / S0 - 1) <= G / 100))\n\ndef cost(p, sig, screen="final", wafer_mult=1.0, N=None):\n    yg = max(y_route(sig, p["G"]), 1e-6)\n    c_cal = p["rate_cal"] * p["t_cal"] / 3600\n    silicon = p["C_wafer"] * wafer_mult / ((N or p["N"]) * p["Y_wafer"])\n    if screen == "final":\n        c = (silicon + p["C_asic"] + p["C_pkg"] + c_cal) / yg\n    else:\n        c = (silicon + p["C_probe"]) / yg + p["C_asic"] + p["C_pkg"] + c_cal\n    return c / p["Y_other"]\n\nbase = {k: v[0] for k, v in INPUTS.items()}; base["N"] = N_GROSS\nREF = "Timed wet etch (KOH)"\n\ndef breakeven(route, p, screen, die=False):\n    sig_ref, sig = ROUTES[REF][0], ROUTES[route][0]\n    n_ref = n_gross(DIE[REF]) if die else None\n    n_r = n_gross(DIE[route]) if die else None\n    c_ref = cost(p, sig_ref, screen, N=n_ref)\n    lo, hi = 0.1, 100.0\n    if cost(p, sig, screen, hi, N=n_r) <= c_ref: return hi\n    for _ in range(60):\n        mid = (lo + hi) / 2\n        if cost(p, sig, screen, mid, N=n_r) > c_ref: hi = mid\n        else: lo = mid\n    return lo\n\ncost_rows = []\nfor G in [5, 10, 15, 20, 25]:\n    p = dict(base); p["G"] = G\n    for screen in ("final", "probe"):\n        for route in ROUTES:\n            sig = ROUTES[route][0]\n            cost_rows.append({"G_pct": G, "screen": screen, "route": route,\n                              "yield_in_range": round(y_route(sig, G), 4),\n                              "cost_same_wafer_price_usd": round(cost(p, sig, screen), 4),\n                              "breakeven_wafer_multiple": round(breakeven(route, p, screen), 3) if route != REF else 1.0,\n                              "die_mm": DIE[route], "dies_per_wafer": n_gross(DIE[route]),\n                              "breakeven_with_route_die_size": round(breakeven(route, p, screen, die=True), 3) if route != REF else 1.0})\nwith open(os.path.join(DATA, "cost_breakeven.csv"), "w", newline="") as fh:\n    w = csv.DictWriter(fh, fieldnames=list(cost_rows[0].keys())); w.writeheader(); w.writerows(cost_rows)\n\np = dict(base); yg = y_route(ROUTES[REF][0], p["G"])\nsilicon = p["C_wafer"] / (p["N"] * p["Y_wafer"])\nback = p["C_asic"] + p["C_pkg"] + p["rate_cal"] * p["t_cal"] / 3600\nsummary = {"N_gross": N_GROSS, "N_gross_2mm_die": n_gross(2.0), "silicon_cost_per_die_usd": round(silicon, 4),\n           "backend_cost_per_die_usd": round(back, 4), "backend_share": round(back / (silicon + back), 3),\n           "calibration_cost_usd": round(p["rate_cal"] * p["t_cal"] / 3600, 4), "S0_mV_per_V": round(S0, 3),\n           "fem_sweep_logfit_exponent": round(-logfit[0], 3)}\n\ntorn = []\nfor route in (REF, "Cavity-SOI, standard (±0.5 µm)"):\n    c0 = cost(base, ROUTES[route][0])\n    for k, (b, lo, hi, src) in INPUTS.items():\n        vals = []\n        for v in (lo, hi):\n            p = dict(base); p[k] = v\n            vals.append(cost(p, ROUTES[route][0]))\n        torn.append({"route": route, "input": k, "base_cost": round(c0, 4), "cost_at_low": round(vals[0], 4),\n                     "cost_at_high": round(vals[1], 4), "low": lo, "high": hi, "source": src})\nwith open(os.path.join(DATA, "cost_tornado.csv"), "w", newline="") as fh:\n    w = csv.DictWriter(fh, fieldnames=list(torn[0].keys())); w.writeheader(); w.writerows(torn)\nwith open(os.path.join(DATA, "cost_inputs.csv"), "w", newline="") as fh:\n    w = csv.writer(fh); w.writerow(["input", "base", "low", "high", "source"])\n    for k, v in INPUTS.items(): w.writerow([k, *v])\ndef y_general(route, G, dop_cv=0.0, shape="normal", seed=11):\n    r = np.random.default_rng(seed)\n    if shape == "normal": t = T0 + r.normal(0, ROUTES[route][0], N_MC)\n    else: t = T0 + r.uniform(-TOL[route], TOL[route], N_MC)\n    s = sens(t) / S0 * (1 + r.normal(0, dop_cv, N_MC))\n    return float(np.mean(np.abs(s - 1) <= G / 100))\n\ndef cost_y(p, yg, screen, wafer_mult=1.0, N=None):\n    c_cal = p["rate_cal"] * p["t_cal"] / 3600\n    silicon = p["C_wafer"] * wafer_mult / ((N or p["N"]) * p["Y_wafer"])\n    if screen == "final": c = (silicon + p["C_asic"] + p["C_pkg"] + c_cal) / yg\n    else: c = (silicon + p["C_probe"]) / yg + p["C_asic"] + p["C_pkg"] + c_cal\n    return c / p["Y_other"]\n\ndef breakeven_y(y_ref, y_r, p, screen):\n    c_ref = cost_y(p, y_ref, screen); lo, hi = 0.1, 100.0\n    if cost_y(p, y_r, screen, hi) <= c_ref: return hi\n    for _ in range(60):\n        mid = (lo + hi) / 2\n        if cost_y(p, y_r, screen, mid) > c_ref: hi = mid\n        else: lo = mid\n    return lo\n\nrob = []\nfor label, kw in (("normal, no doping spread", {}), ("normal + doping CV 2 %", {"dop_cv": 0.02}),\n                  ("normal + doping CV 4 %", {"dop_cv": 0.04}), ("uniform over full tolerance", {"shape": "uniform"})):\n    for G in (5, 10, 20):\n        p = dict(base); p["G"] = G\n        yref = y_general(REF, G, **kw)\n        for route in ROUTES:\n            y = y_general(route, G, **kw)\n            rob.append({"scenario": label, "G_pct": G, "route": route, "yield_in_range": round(y, 4),\n                        "breakeven_final": round(breakeven_y(yref, y, p, "final"), 3) if route != REF else 1.0,\n                        "breakeven_probe": round(breakeven_y(yref, y, p, "probe"), 3) if route != REF else 1.0})\nwith open(os.path.join(DATA, "process_robustness.csv"), "w", newline="") as fh:\n    w = csv.DictWriter(fh, fieldnames=list(rob[0].keys())); w.writeheader(); w.writerows(rob)\n\njson.dump(summary, open(os.path.join(DATA, "cost_summary.json"), "w"), indent=1)\nprint(json.dumps(summary, indent=1))\n', 'data/sweep_su.csv': 't_um,w_centre_um,sigma_edge_MPa,sxL_MPa,vout_mV_per_V,sens_mV_per_V_per_kPa\n9.0,1.83054,94.696,101.806,73.7039,0.737039\n10.0,1.34575,78.988,81.992,59.3945,0.593945\n11.0,1.01949,66.692,67.37,48.824,0.48824\n12.0,0.79169,56.915,56.277,40.7962,0.407962\n13.0,0.62771,49.034,47.668,34.5588,0.345588\n14.0,0.50659,42.603,40.854,29.619,0.29619\n15.0,0.41512,37.299,35.373,25.6429,0.256429\n16.0,0.34472,32.882,30.9,22.3973,0.223973\n17.0,0.28961,29.171,27.203,19.7154,0.197154\n', 'data/public/cost_model_inputs.csv': 'name,base,low,high,unit,kind,source,match_to_this_study\nC_wafer,800,400,1600,USD per processed 150 mm wafer,published model value,"Lawes (2007), Microsyst. Technol. 13:85-95: CMOS+MEMS accelerometer on 150 mm wafers, 10,000 wafers/yr: $750 electronics + ~$30 DRIE + $20 blank = ~$800","Same wafer size as the die count used here (150 mm). Different device (accelerometer), and 2007 prices, so the range is x0.5 to x2"\nY_wafer,0.9,0.8,0.97,fraction,assumed,,\nC_asic,0.15,0.05,0.4,USD per sensor,assumed,,\nC_pkg,0.1,0.05,0.3,USD per sensor,published estimate,"Fitzgerald (2021), NCCAVS webinar: molded pressure-sensor package under $0.10","Same device type (pressure sensor package)"\nrate_cal,25,10,60,USD per hour of chamber time,assumed,,\nt_cal,33.75,16.875,50.625,s per sensor,published measurement,"Pavlin & Novak (2008), Sens. Actuators A 141:34-42: 3 pressures x 3 temperatures, 128 sensors in about 72 min","Same sensor type (piezoresistive pressure sensor) and same need for multi-temperature calibration; high value = batch two-thirds full"\nY_other,0.97,0.93,0.99,fraction,assumed,,\nC_probe,0.03,0.01,0.08,USD per die,assumed,,\nG,10,5,20,percent,assumed,,\n', 'data/public/lawes_2007_values.csv': 'item,value,unit,context\nblank wafer,25,USD,model input example\nmask (optical aligner),700,USD per mask,Table 2\nmask (optical stepper),1500,USD per mask,Table 2\nwet bulk micromachining layer,2.97,USD per wafer (variable incl. 2 USD consumables),Table 2\nthree-poly surface micromachining,652,USD per 200 mm wafer,"10,000 wafers/yr"\nCMOS electronics,750,USD per 150 mm wafer,"accelerometer example, 10,000 wafers/yr"\nCMOS + MEMS accelerometer total,800,USD per 150 mm wafer,"750 + ~30 DRIE + 20 blank"\naccelerometer chip,0.22,USD per 4.5 mm2 chip,"100 % yield, 3,697 chips per 150 mm wafer"\npackaging share,more than 50,percent of device cost,"rises to about 70 % for a 2 mm2 chip"\n', 'data/public/market_prices.csv': 'part,maker,type,package,quantity,unit_price_usd,distributor,source,accessed\nBMP390,Bosch Sensortec,piezoresistive barometric pressure sensor with ASIC,LGA-10 2.0 x 2.0 x 0.75 mm,1,3.58,Mouser,https://www.mouser.com/ProductDetail/Bosch-Sensortec/BMP390?qs=QNEnbhJQKvYQVfvRMgo2YA%3D%3D,2026-10-31\nBMP390,Bosch Sensortec,piezoresistive barometric pressure sensor with ASIC,LGA-10 2.0 x 2.0 x 0.75 mm,100,2.67,Mouser,same,2026-10-31\nBMP390,Bosch Sensortec,piezoresistive barometric pressure sensor with ASIC,LGA-10 2.0 x 2.0 x 0.75 mm,1000,2.38,Mouser,same,2026-10-31\nBMP390,Bosch Sensortec,piezoresistive barometric pressure sensor with ASIC,LGA-10 2.0 x 2.0 x 0.75 mm,10000,2.16,Mouser,same,2026-10-31\n', 'data/public/mpw_prices_2026.csv': 'process,unit,price_standard_eur,price_academic_eur,area_mm2,eur_per_mm2_standard,note,source,accessed\nScience SOI (MEMSCAP SOIMUMPs successor),block 9 x 9 mm,13000,10000,81,160,prototype run,https://europractice-ic.com/schedules-prices-2026/,2026-10-31\nScience Piezo,block 9 x 9 mm,13000,10000,81,160,prototype run,https://europractice-ic.com/schedules-prices-2026/,2026-10-31\nScience Poly (PolyMUMPs successor),block 10 x 10 mm,19500,15000,100,195,prototype run,https://europractice-ic.com/schedules-prices-2026/,2026-10-31\nX-FAB XMB10 MEMS (core),per mm2 (min. 10 mm2),1253,1168,1,1253,50 dies included,https://europractice-ic.com/schedules-prices-2026/,2026-10-31\n', 'data/public/thickness_control.csv': 'route,sigma_base_um,sigma_conservative_um,band_um,die_mm,what_the_source_gives,source\nTimed wet etch (KOH),0.5,1.0,1.0,2.0,"die-to-die variation about +-1 um within a wafer, 10-14 um KOH membranes (base: +-2 sigma; conservative: +-1 sigma)",Singh et al. (2014) IEEE Trans. Semicond. Manuf. 27(3):410-416\nElectrochemical etch stop,0.2,0.3,0.6,2.0,"membrane thickness standard deviation 0.2 um; band = +-3 sigma",Kloeck et al. (1989) IEEE Trans. Electron Devices 36(4):663-669\n"Cavity-SOI, standard (±0.5 µm)",0.16666666666666666,0.25,0.5,1.2,"device layer typically +-0.5 um (base: +-3 sigma; conservative: +-2 sigma)",Okmetic C-SOI product page (accessed Oct 2026)\n"Cavity-SOI, as quoted by Su (±1 µm)",0.3333333333333333,0.5,1.0,1.2,"cavity-SOI thickness variation +-1 um after grinding (base: +-3 sigma; conservative: +-2 sigma)","Su et al. (2019) J. Sensors 5408268, Section 4"\n"Cavity-SOI, EC-SOI (±0.2 µm)",0.06666666666666667,0.1,0.2,1.2,"down to +-0.2 um (base: +-3 sigma; conservative: +-2 sigma)",Okmetic C-SOI product page (accessed Oct 2026)\n"Silicon-on-nothing (<±0.2 µm)",0.06666666666666667,0.1,0.2,1.2,"epitaxial membrane less than +-0.2 um, claimed by the authors (base: +-3 sigma; conservative: +-2 sigma)","Su et al. (2019) J. Sensors 5408268, Section 4"\n', 'data/public/thickness_scenarios.csv': 'route,sigma_um,note\nAPSM / MIS scenario,0.1,no published thickness data for Bosch APSM (Boehringer et al. 2012) or MIS (Jiao et al. 2020)\nAPSM / MIS scenario,0.3,same\nAPSM / MIS scenario,0.6,same\n', 'data/public/validation_points.csv': 'quantity,value,unit,conditions,source,how_it_is_used\ncentre deflection of the 13 um SON membrane,0.64,um,"600 x 600 um, 13 um, 100 kPa, ANSYS","Su et al. (2019), Section 3.1, Fig. 2b",FEM check\nsensitivity at 0 C,0.79,mV/kPa,"SON sensor, supply voltage not given","Su et al. (2019), Section 4",order-of-magnitude check of piezoresistive coefficients\nsensitivity at 40 C,0.72,mV/kPa,same,"Su et al. (2019), Section 4",same\nsensitivity at 80 C,0.67,mV/kPa,same,"Su et al. (2019), Section 4",same\nresistor sheet resistance,200-500,ohm/sq,light boron implant,"Su et al. (2019), Section 3.4",doping level of the reference sensor\nmeasured sensitivity 10 um membranes,86.7-96.6,mV/V/bar,"5 V, 50 kPa, 5 dies per wafer, thickness +-1 um","Singh et al. (2014), Table III",FEM check\nmeasured sensitivity 12 um membranes,71.9-84.5,mV/V/bar,same,"Singh et al. (2014), Table III",FEM check\nmeasured sensitivity 14 um membranes,50.5-54.5,mV/V/bar,same,"Singh et al. (2014), Table III",FEM check\nsensitivity spread with EC etch stop,4,% (1 s.d.),"devices on the same wafer, EC etch-stop membranes","Kloeck et al. (1989), abstract",checks the Monte Carlo spread for the etch-stop route\nsensitivity range without etch stop,about 2x,ratio,devices on the same wafer,"Kloeck et al. (1989), abstract",supports the conservative case for timed etching\n'}
for name, text in FILES.items():
    os.makedirs(os.path.dirname(name) or '.', exist_ok=True)
    open(name, 'w', encoding='utf-8').write(text)
print(len(FILES), 'files written')

**Solver check**, isotropic plate E = 169 GPa, ν = 0.3, 600 µm, 13 µm, 100 kPa. Plate theory gives 0.480 µm, the paper reports w_centre_um = 0.4847.

In [ ]:
!python membrane_fem.py --iso 169 0.3 --nxy 12 --nz 2

**Reference membrane**, anisotropic Si, 13 µm, 100 kPa. Expected: clamp 0.5743 µm and 36.34 mV/V, frame 0.6277 µm and 34.56 mV/V (Su et al. 2019 report 0.64 µm from ANSYS).

In [ ]:
!python membrane_fem.py --nxy 12 --nz 2
!python membrane_fem.py --nxy 12 --nz 2 --frame 150 100

**Singh et al. (2014) 12 µm membrane**, 1097 µm wide, 50 kPa, stress averaged over the 1 µm resistor depth. Expected 34.89 mV/V, which is 69.8 mV/V/bar. Their FEM 70.5, measured 71.9–84.5 mV/V/bar. Takes 2–4 min.

In [ ]:
!python membrane_fem.py --nxy 12 --nz 2 --t 12 --side 1097 --p 50 --pi 3 -1 80 --resL 60 10 10 --resT 60 10 20 --frame 300 600 --depth 1

**Large deflection**, same coarse mesh for both. Expected linear 0.5702 µm / 36.33 mV/V, nonlinear 0.5697 µm / 36.52 mV/V.

In [ ]:
!python membrane_fem.py --nxy 8 --nz 1
!python nonlinear_check.py --nxy 8 --nz 1 --steps 1

**Monte Carlo and cost model**. Expected at ±10 %: timed etch keeps 79 % of chips, break-even 2.6× (final test) and 1.35× (wafer probe).

In [ ]:
!python process_cost.py
import pandas as pd
display(pd.read_csv('data/process_montecarlo.csv'))
display(pd.read_csv('data/cost_breakeven.csv').query('G_pct == 10'))
display(pd.read_csv('data/process_robustness.csv').query('G_pct == 10'))